# Task 3 ensemble

Rating prediction with timestamps. The Task 1 models (timestamps ignored) plus five temporal models:

- **time-aware bias model**: time-bin, user-drift and user-period biases in closed form (PML Sec. 7.2.2, Koren 2009)
- **timeSVD++**: SVD++ on top of the time-aware bias model
- **time-decayed item kNN**: neighbours rated close to the asked day count more
- **temporal zero model**: P(0) also from the zeros the user gave around that day
- **time-aware factorization machine**: the Task 1 factorization machine + one-hot calendar fields of the timestamp (month and year; sweep 14 Nov)

Every sub-model cell trains one model and stores it in `assignment1_2026/trained_models/task3/<model>.pt`.
The blend cells only need the stored files, so after a kernel restart nothing has to be retrained.
All hyperparameters: `lib/config.py`, Task 3 section.

In [ ]:
%load_ext autoreload
# --full: reload whole modules. The default only re-runs the changed lines of a module, so a setting computed from an
# edited one (e.g. TASK1_BLENDED_MODELS from TASK1_BLEND_MODEL_FLAGS in lib/config.py) would keep its old value
%autoreload 2 --full

# assignment1.py and lib/ live next to this notebook (in submission/)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import torch

import assignment1
import lib.config as model_configs        # settings read when a cell runs -> edits of config.py apply without a restart
from lib.models.bias_model import load_bias_model, store_bias_model
from lib.models.blender import load_blender, store_blender
from lib.models.item_knn import load_item_knn, store_item_knn
from lib.models.SVDpp import load_svd, load_svdpp, store_svd, store_svdpp
from lib.models.time_bias_model import load_time_bias_model, store_time_bias_model
from lib.models.time_factorization_machine import load_time_factorization_machine, store_time_factorization_machine
from lib.models.time_item_knn import load_time_item_knn, store_time_item_knn
from lib.models.time_svdpp import load_time_svdpp, store_time_svdpp
from lib.models.time_zero_model import load_time_zero_model, store_time_zero_model
from lib.models.zero_model import load_zero_model, store_zero_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

### Data and validation
Ratings $\mathcal R = \{(u, i, r_{ui})\}$, $r_{ui} \in [0, 5]$, split at random into 90 % training $\mathcal T$ and 10 % validation $\mathcal V$ (`VALIDATION_FRACTION`, `SPLIT_SEED`).
Every sub-model: fit on $\mathcal T$ $\rightarrow$ predictions $\hat r^{(m)}_{ui}$ for $\mathcal V$ (input of the blender) $\rightarrow$ refit on $\mathcal T \cup \mathcal V$ $\rightarrow$ test predictions.

$$\mathrm{MSE} = \frac{1}{|\mathcal V|}\sum_{(u,i)\in\mathcal V}\big(r_{ui}-\hat r_{ui}\big)^2$$

Every rating also has its day $t$ (days since 2000-01-01; impossible timestamps are clipped to 2000 - 2017). The test file has $t$ as well.
The time-aware factorization machine uses the full timestamp $\tau$ (unix seconds, UTC, clipped the same way).

In [ ]:
# training file + test file with timestamps -> task3.test_days / task3.test_timestamps are set
task3 = assignment1.load_task3()

## Sub-models without time

### Bias model (PML Eq. 5.10 without factors)
$$\hat r_{ui} = \mu + b_u + b_i$$
$$\min_{b}\ \sum_{(u,i)\in\mathcal T}\big(r_{ui}-\hat r_{ui}\big)^2 + \lambda_{b_u}\sum_u b_u^2 + \lambda_{b_i}\sum_i b_i^2$$
Closed form, alternating for 10 rounds ($U_i$ = users who rated $i$, $I_u$ = items rated by $u$,
$\lambda_{b_u}$ = `user_bias_regularization`, $\lambda_{b_i}$ = `item_bias_regularization`):
$$b_i \leftarrow \frac{\sum_{u\in U_i}\left(r_{ui}-\mu-b_u\right)}{|U_i|+\lambda_{b_i}}, \qquad b_u \leftarrow \frac{\sum_{i\in I_u}\left(r_{ui}-\mu-b_i\right)}{|I_u|+\lambda_{b_u}}$$

In [ ]:
# bias model: mu + b_u + b_i in closed form (seconds)
bias_model_result = assignment1.train_task3_bias_model(task3)
store_bias_model(bias_model_result, "task3")

### SVD / latent factor model (PML Eq. 5.10)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top \mathbf p_u$$
$$\min_{b,\mathbf p,\mathbf q}\ \sum_{(u,i)\in\mathcal T}\big(r_{ui}-\hat r_{ui}\big)^2 + \lambda_{b_u}\sum_u b_u^2 + \lambda_{b_i}\sum_i b_i^2 + \lambda_f\Big(\sum_u\lVert\mathbf p_u\rVert^2+\sum_i\lVert\mathbf q_i\rVert^2\Big)$$
(PML Eq. 5.12 with the bias weight $\lambda_1$ split into $\lambda_{b_u}$ and $\lambda_{b_i}$; $\lambda_f = \lambda_2$.)
Adam on mini-batches (each parameter's penalty is spread over the ratings that use it), biases start at the closed-form bias model,
early stopping on the validation MSE, then a refit on all ratings for the best number of epochs.

In [ ]:
# SVD (a few minutes on the RTX 5080)
svd_result = assignment1.train_task3_svd(task3)
store_svd(svd_result, "task3")

### SVD++ (Koren 2008; implicit feedback as in FISM, PML Sec. 5.3.2)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top\Big(\mathbf p_u + |N(u)|^{-1/2}\sum_{j\in N(u)}\mathbf y_j\Big)$$
$N(u)$ = the items $u$ rated in the data the model trains on; $\mathbf y_j$ is regularised with $\lambda_f$ like $\mathbf q_j$. Training as for SVD.

In [ ]:
# SVD++ (a few minutes on the RTX 5080)
svdpp_result = assignment1.train_task3_svdpp(task3)
store_svdpp(svdpp_result, "task3")

### Item-item kNN on the bias model's residuals (PML Ch. 4; Koren 2010)
Baseline $b_{ui} = \mu + b_u + b_i$ (closed form), residuals $e_{uj} = r_{uj} - b_{uj}$.

Shrunk cosine similarity ($n_{ij}$ = users who rated both, $\lambda_s$ = `similarity_shrinkage`), only the $k$ most similar items $N_k(i)$ are kept:
$$s_{ij} = \frac{\sum_u e_{ui}\,e_{uj}}{\lVert\mathbf e_{\cdot i}\rVert\,\lVert\mathbf e_{\cdot j}\rVert}\cdot\frac{n_{ij}}{n_{ij}+\lambda_s}$$
$$\hat r_{ui} = b_{ui} + \frac{\sum_{j\in N_k(i)\cap I_u} s_{ij}\,e_{uj}}{\sum_{j\in N_k(i)\cap I_u} s_{ij} + \lambda_p}$$

In [ ]:
# item-item kNN on the bias model's residuals (CPU, about a minute)
item_knn_result = assignment1.train_task3_item_knn(task3)
store_item_knn(item_knn_result, "task3")

### Zero model: $P(r_{ui} = 0)$ (logistic regression, PML Sec. 3.3)
$$P(r_{ui}=0) = \sigma\big(w_0 + \mathbf w^\top \mathbf x_{ui}\big), \qquad \mathbf x_{ui} = \Big[\operatorname{logit}\tilde z_u,\ \operatorname{logit}\tilde z_i,\ \log(1+n_u),\ \log(1+n_i)\Big]$$
$$\tilde z_u = \frac{z_u + s_u\,\pi}{n_u + s_u}$$
$z_u$ = zeros of user $u$, $n_u$ = ratings of $u$ (likewise for item $i$ with $s_i$), $\pi$ = global share of zeros, $s$ = prior strength.
Training rows leave their own rating out; features standardised. The output is a probability, an input of the blender (not a rating).

In [ ]:
# zero model: P(rating = 0) (CPU, seconds)
zero_model_result = assignment1.train_task3_zero_model(task3)
store_zero_model(zero_model_result, "task3")

## Temporal sub-models

### Time-aware bias model (PML Sec. 7.2.2, Eq. 7.8 - 7.12; Koren 2009)
$$\hat r_{ui}(t) = \mu + b_u + b_i + b_{\mathrm{bin}(t)} + b_{i,\mathrm{bin}(t)} + \alpha_u\,\mathrm{dev}_u(t) + \sum_{s\in S} b_{u,\lfloor t/s\rfloor}$$
$$\mathrm{dev}_u(t) = \operatorname{sign}(t-\bar t_u)\,|t-\bar t_u|^{\beta}$$
30 bins = quantiles of the training days; $S = \{1, 7, 30, 90, 365\}$ days (user-period biases); $\beta = 0.4$; $\bar t_u$ = the user's mean rating day.

Closed form, alternating for 10 rounds. Every bias table $g$ (each row = a group of ratings, e.g. all ratings of one user in one week):
$$b_g \leftarrow \frac{\sum_{(u,i,t)\in g}\big(r_{ui}-\hat r_{ui}(t)+b_g\big)}{|g|+\lambda_g}, \qquad \alpha_u \leftarrow \frac{\sum_{i\in I_u}\mathrm{dev}_u(t_{ui})\,\big(r_{ui}-\hat r_{ui}(t_{ui})+\alpha_u\,\mathrm{dev}_u(t_{ui})\big)}{\sum_{i\in I_u}\mathrm{dev}_u(t_{ui})^2+\lambda_\alpha}$$
$\lambda_g = 3$ for $b_u, b_i, b_{\mathrm{bin}}$; $\lambda_g = 50$ for the item-bin and user-period biases; $\lambda_\alpha = 3000$.

In [ ]:
# time-aware bias model: mu + b_u + b_i + b_bin(t) + b_{i,bin(t)} + alpha_u dev_u(t) + b_{u,period(t)} in closed form (CPU, seconds)
time_bias_model_result = assignment1.train_task3_time_bias_model(task3)
store_time_bias_model(time_bias_model_result, "task3")

### timeSVD++ (Koren 2009; PML Eq. 7.13)
$$\hat r_{ui}(t) = \underbrace{\mu + b_u + b_i + b_{\mathrm{bin}(t)} + b_{i,\mathrm{bin}(t)} + \alpha_u\,\mathrm{dev}_u(t) + \textstyle\sum_{s} b_{u,\lfloor t/s\rfloor}}_{\text{time-aware bias model (closed form)}} + \mathbf q_i^\top\Big(\mathbf p_u + |N(u)|^{-1/2}\sum_{j\in N(u)}\mathbf y_j\Big)$$
All biases start at the closed-form time-aware bias model; $b_u$, $b_i$ and the factors are trained with Adam as in SVD++, the temporal biases stay fixed
(`train_temporal_biases=False`). Optional factor drift: $\mathbf p_u(t) = \mathbf p_u + \boldsymbol\alpha^{(p)}_u\,\mathrm{dev}_u(t)$.

In [ ]:
# timeSVD++: SVD++ on top of the time-aware bias model (the slowest sub-model, GPU)
time_svdpp_result = assignment1.train_task3_time_svdpp(task3)
store_time_svdpp(time_svdpp_result, "task3")

### Time-decayed item-item kNN (Koren 2009, temporal neighbourhood; Ding & Li 2005, PML Sec. 7.3.1)
Same neighbourhoods $s_{ij}$ as the item kNN; neighbours the user rated close to the asked day count more:
$$\hat r_{ui}(t) = b_{ui} + \frac{\sum_{j\in N_k(i)\cap I_u} s_{ij}\,d_{uj}(t)\,e_{uj}}{\sum_{j\in N_k(i)\cap I_u} s_{ij}\,d_{uj}(t) + \lambda_p}, \qquad d_{uj}(t) = \varphi + (1-\varphi)\,e^{-|t - t_{uj}|/\tau}$$
$\tau = 365$ days, floor $\varphi = 0.5$.

In [ ]:
# time-decayed item kNN: neighbour weight * (floor + (1 - floor) exp(-|t - t_uj| / time_decay_days)) (CPU, about a minute)
time_item_knn_result = assignment1.train_task3_time_item_knn(task3)
store_time_item_knn(time_item_knn_result, "task3")

### Temporal zero model
The features of the zero model plus, for every window $w \in \{0, 30\}$ days around $t$ (the user's other ratings only):
$$\operatorname{logit}\frac{z_u(t,w) + s\,\pi}{n_u(t,w) + s}, \qquad \log\big(1 + n_u(t,w)\big)$$
$n_u(t,w)$ / $z_u(t,w)$ = the user's ratings / zeros within $\pm w$ days of $t$, $s$ = `window_prior_strength`.

In [ ]:
# temporal zero model: P(rating = 0) + the user's zeros on the same day / within a month (CPU, seconds)
time_zero_model_result = assignment1.train_task3_time_zero_model(task3)
store_time_zero_model(time_zero_model_result, "task3")

### Time-aware factorization machine (Rendle 2010; PML Sec. 6.1, Eq. 6.1 and Exercise 6.1)
The Task 1 factorization machine plus one-hot calendar fields of the timestamp $\tau$ (UTC):
$$\hat r(\mathbf x) = \mu + \sum_f w_f x_f + \sum_{f<g}\langle\mathbf v_f,\mathbf v_g\rangle\,x_f x_g, \qquad \text{fields: } u,\ i,\ a_u,\ a_i,\ h(\tau),\ d(\tau),\ m(\tau),\ y(\tau)$$
$h$ = hour of day (24), $d$ = weekday (7), $m$ = month (12), $y$ = year (first to last year of the training ratings).
E.g. $\langle\mathbf v_i,\mathbf v_{m(\tau)}\rangle$ = a book's seasonal popularity, $\langle\mathbf v_u,\mathbf v_{d(\tau)}\rangle$ = a user's weekday habits.
The calendar features are unregularised (shared by many ratings), like the buckets. `calendar_fields` in `lib/config.py` selects the fields.
Note: every timestamp lies between 05:00 and 09:59 UTC and the window moves with US daylight saving time (dates + an offset),
so the hour field carries little real signal. The sweep of 14 Nov kept month + year (0.8941 vs 0.8967 with all four).

In [ ]:
# time-aware factorization machine: + the calendar fields of the timestamp (month, year) (a few minutes on the RTX 5080)
time_factorization_machine_result = assignment1.train_task3_time_factorization_machine(task3)
store_time_factorization_machine(time_factorization_machine_result, "task3")

## Blend

### Blender input (stacking)
For every validation rating $(u,i)\in\mathcal V$: the predictions $\hat r^{(1)}_{ui},\dots,\hat r^{(M)}_{ui}$ of the sub-models in `blended_models`
(= the models set to `True` in `TASK3_BLEND_MODEL_FLAGS` in `lib/config.py`),
made by models that never saw $\mathcal V$. For the test rows: the predictions of the refitted sub-models.

In [ ]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "bias_model": load_bias_model,
    "svd": load_svd,
    "svdpp": load_svdpp,
    "item_knn": load_item_knn,
    "zero_model": load_zero_model,
    "time_bias_model": load_time_bias_model,
    "time_svdpp": load_time_svdpp,
    "time_item_knn": load_time_item_knn,
    "time_zero_model": load_time_zero_model,
    "time_factorization_machine": load_time_factorization_machine,
}
task3_model_results = {model_name: sub_model_loaders[model_name]("task3") for model_name in model_configs.TASK3_SUBMITTED_BLENDER.blended_models}

### Ridge blend (linear stacking)
$$\hat r_{ui} = \operatorname{clip}_{[0,5]}\Big(\theta_0 + \sum_{m=1}^{M}\theta_m\,\hat r^{(m)}_{ui}\Big), \qquad \min_{\boldsymbol\theta}\ \sum_{(u,i)\in\mathcal V}\big(r_{ui} - \theta_0 - \boldsymbol\theta^\top\hat{\mathbf r}_{ui}\big)^2 + \alpha\lVert\boldsymbol\theta\rVert^2$$
Score: 5-fold out-of-fold MSE on $\mathcal V$; the final blender is fitted on all of $\mathcal V$.

In [ ]:
# ridge regression: learned weighted average of the sub-model predictions
ridge_blend_result = assignment1.train_task3_blender(task3, task3_model_results, model_configs.TASK3_RIDGE_BLENDER)
store_blender(ridge_blend_result, "task3")

### LightGBM blend (gradient-boosted regression trees)
$$\hat r_{ui} = \operatorname{clip}_{[0,5]}\Big(\sum_{t=1}^{T}\eta\,f_t(\mathbf x_{ui})\Big), \qquad \mathbf x_{ui} = \big[\hat r^{(1)}_{ui},\dots,\hat r^{(M)}_{ui},\ \log(1+n_u),\ \log(1+n_i)\big]$$
Each tree $f_t$ is fitted to the negative gradient of the squared error of the trees before it ($T$ = `lightgbm_n_estimators`, $\eta$ = learning rate).
Score: 5-fold out-of-fold MSE on $\mathcal V$.

Task 3 time features (counted in the ratings the sub-models had): $\log(1+n_u^{\text{same day}}(t))$, $\log(1+n_u^{\pm 30\text{ days}}(t))$, $\log(1+n_u^{<t})$, the item's age at $t$ in years, and $t$ in years since 2000.

In [ ]:
# LightGBM on the sub-model predictions + #ratings of user and item + when the rating happened (time features)
lightgbm_blend_result = assignment1.train_task3_blender(task3, task3_model_results, model_configs.TASK3_LIGHTGBM_BLENDER)
store_blender(lightgbm_blend_result, "task3")

### Submission
The test predictions $\hat r_{ui}$ of the blender chosen in `TASK<n>_SUBMITTED_BLENDER` go into the `rating` column.

In [ ]:
# the submitted blend (TASK3_SUBMITTED_BLENDER in lib/config.py) writes output/task_3_predictions.csv
# -> set it to whichever blend printed the lower out-of-fold MSE above
submitted_blend_result = load_blender("task3", model_configs.TASK3_SUBMITTED_BLENDER.method)
assignment1.write_task3_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK3_PREDICTION_FILE)